In [72]:
from datetime import datetime

#!pip install openmeteo_requests

import openmeteo_requests


def open_forecast():
    openmeteo = openmeteo_requests.Client()
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 59.9386,  # for St.Petersburg
        "longitude": 30.3141,  # for St.Petersburg
        "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
        "wind_speed_unit": "ms",
        "timezone": "Europe/Moscow",
    }

    response = openmeteo.weather_api(url, params=params)[0]

    # The order of variables needs to be the same as requested in params->current!
    current = response.Current()
    current_temperature_2m = current.Variables(0).Value()
    current_apparent_temperature = current.Variables(1).Value()
    current_rain = current.Variables(2).Value()
    current_wind_speed_10m = current.Variables(3).Value()

    print(
        f"Current time: {datetime.fromtimestamp(current.Time()+response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}"
    )
    print(f"Current temperature: {round(current_temperature_2m, 0)} C")
    print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
    print(f"Current rain: {current_rain} mm")
    print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")


class IncreaseSpeed:
    """
    Iterator for increasing the speed with the default step of 10 km/h

    Constructor params:
        current_speed: a value to start with, km/h
        max_speed: a maximum possible value, km/h
        step: a parameter to vary the value to increase by

    Make sure your iterator is not exceeding the maximum allowed value
    """

    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step
        assert step >= 0

    def __iter__(self):
        return self

    def __next__(self):
        self.current_speed += self.step
        if self.current_speed <= self.max_speed:
            print(f"INFO: Speed increased by {self.step}")
            return self.current_speed
        if self.current_speed > self.max_speed:
            print(f"INFO: Reached max speed of {self.max_speed}")
            return self.max_speed


class DecreaseSpeed:
    """
    Iterator for decreasing the speed with the default step of 10 km/h

    Constructor params:
        current_speed: a value to start with, km/h

    Make sure your iterator is not going below zero
    """

    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step
        assert step >= 0

    def __iter__(self):
        return self

    def __next__(self):
        self.current_speed -= self.step
        if (self.current_speed >= self.min_speed) and (self.current_speed >= 0):
            print(f"INFO: Speed decreased by {self.step}")
            return self.current_speed
        if self.current_speed < 0:
            print(f"INFO: Speed reached 0")
            return 0


class Car:
    """
    Car class.
    Has a class variable for counting total amount of cars on the road (increased by 1 upon instance initialization).

    Constructor params:
        max_speed: a maximum possible speed, km/h
        current_speed: current speed, km/h (0 by default)
        state: reflects if the Car is in the parking or on the road

    Methods:
        accelerate: increases the speed using IncreaseSpeed() iterator either once or gradually to the upper_border
        brake: decreases the speed using DecreaseSpeed() iterator either once or gradually to the lower_border
        parking: if the Car is not already in the parking, removes the Car from the road
        total_cars: show the total amount of cars on the road
        show_weather: shows the current weather conditions
    """

    total_cars_num: int = 0

    def __init__(self, max_speed: int, current_speed: int = 0):
        self.max_speed: int = max_speed
        self.current_speed: int = current_speed
        self.state: bool = bool(current_speed)
        if self.state:
            Car.total_cars_num += 1

    def accelerate(self, upper_border=None | int, step=10):
        old_speed = self.current_speed
        if not self.state and step:
            self.state = True
            Car.total_cars_num += 1

        transmission = IncreaseSpeed(self.current_speed, self.max_speed, step)
        change_gear = iter(transmission)
        if upper_border is not None:
            if upper_border > self.max_speed:
                upper_border = self.max_speed
            assert upper_border >= self.current_speed
            
            while self.current_speed < upper_border:
                self.current_speed = next(change_gear)
                
        else:
            self.current_speed = next(change_gear)

        print(f"INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")
        return

    def brake(self, lower_border=None | int, step=10):
        old_speed = self.current_speed
        transmission = DecreaseSpeed(self.current_speed, 0, step)
        change_gear = iter(transmission)
        if lower_border is not None:
            if lower_border < 0:
                lower_border = 0
            assert lower_border <= self.current_speed
            while self.current_speed > lower_border:
                self.current_speed = next(change_gear)
        else:
            self.current_speed = next(change_gear)
        print(f"INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")
        return

    def parking(self):
        if not self.state:
            print("INFO: The car is already parked")
            return

        if self.current_speed > 0:
            self.brake(self, 0)

        self.state = False
        Car.total_cars_num -= 1
        print("INFO: The car has been parked")
        return

    def total_cars():
        print(f"Number of cars on the road: {Car.total_cars_num}")
        return Car.total_cars_num

    @staticmethod
    def show_weather():
        open_forecast()



In [73]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Number of cars on the road: 2
Total cars on road: 2


In [74]:
car1.accelerate(100)

INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: The speed of this car has been decreased from 20 to 100


In [75]:
car2.accelerate(50)

INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: The speed of this car has been decreased from 30 to 50


In [76]:
print("Speed of car 1:", car1.current_speed)
print("Speed of car 2:", car2.current_speed)

Speed of car 1: 100
Speed of car 2: 50


In [77]:
car1.brake(10)

INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: The speed of this car has been decreased from 100 to 10


In [78]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: Speed decreased by 10
INFO: The speed of this car has been decreased from 50 to 0
Number of cars on the road: 2
Total cars on road: 2
INFO: The car has been parked
Number of cars on the road: 1
Total cars on road: 1


In [79]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: The speed of this car has been decreased from 0 to 80
Current time: 2026-03-25 05:45:00 GMT+3
Current temperature: 5.0 C
Current apparent_temperature: 1.0 C
Current rain: 0.0 mm
Current wind_speed: 5.0 m/s
Number of cars on the road: 2
Total cars on road: 2


In [80]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increased by 10
INFO: The speed of this car has been decreased from 0 to 10
Number of cars on the road: 3
Total cars on road: 3


In [81]:
Car.show_weather()

Current time: 2026-03-25 05:45:00 GMT+3
Current temperature: 5.0 C
Current apparent_temperature: 1.0 C
Current rain: 0.0 mm
Current wind_speed: 5.0 m/s
